<a href="https://colab.research.google.com/github/nathanleunggg/Number_Guessing_Game/blob/main/Text_Analysis_Basics_(edited).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Course: BA820 - Unsupervised and Unstructured ML**

**Notebook created by: Mohannad Elhamod**

# Dataset

For illustration purposes, we will use a UCI dataset of Youtube comments along with their classification as ham or spam.

In [1]:
pip install ucimlrepo pandas

In [2]:
from ucimlrepo import fetch_ucirepo

data = fetch_ucirepo(id=380)

text_df = data.data.features[["CONTENT"]]
class_df = data.data.targets



In [3]:
text_df

,CONTENT
0,"Huh, anyway check out this you[tube] channel: ..."
1,Hey guys check out my new channel and our firs...
2,just for test I have to say murdev.com
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿
4,watch?v=vtaRGgvGtWQ Check this out .﻿
...,...
1951,I love this song because we sing it at Camp al...
1952,I love this song for two reasons: 1.it is abou...
1953,wow
1954,Shakira u are so wiredo


In [4]:
class_df

,CLASS
0,1
1,1
2,1
3,1
4,1
...,...
1951,0
1952,0
1953,0
1954,0


In [5]:
# How many unique comments?
text_df["CONTENT"].nunique()

1760

There are many duplicate entries. They inflate the dataset. Let's remove them

In [6]:
# Combine text and labels
df = text_df.join(class_df)

# Drop duplicate messages based on CONTENT
df = df.drop_duplicates(subset=["CONTENT"])

# Split back
text_df = df[["CONTENT"]]
class_df = df[class_df.columns]

# Basic Regex and Text Cleaning and Manipulation

If you think your model may be confused by the same word appearing in different cases (e.g., Hello vs. hello), you may want to convert all text to lowercase.

In [7]:
import pandas as pd

# we can lower case
text_df_modified = pd.DataFrame(text_df["CONTENT"].str.lower())
text_df_modified

,CONTENT
0,"huh, anyway check out this you[tube] channel: ..."
1,hey guys check out my new channel and our firs...
2,just for test i have to say murdev.com
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿
4,watch?v=vtarggvgtwq check this out .﻿
...,...
1950,well done shakira
1951,i love this song because we sing it at camp al...
1952,i love this song for two reasons: 1.it is abou...
1954,shakira u are so wiredo


If your model may benefit from unifying formal "you" and slang "u" into one term, you may want to convert one to another.

In [8]:
# replace u with you
text_df_modified = pd.DataFrame(text_df["CONTENT"].str.replace(r"\bu\b", "you", regex=True)) # \b is for word boundaroes
text_df_modified

,CONTENT
0,"Huh, anyway check out this you[tube] channel: ..."
1,Hey guys check out my new channel and our firs...
2,just for test I have to say murdev.com
3,me shaking my sexy ass on my channel enjoy ^_^ ﻿
4,watch?v=vtaRGgvGtWQ Check this out .﻿
...,...
1950,well done shakira
1951,I love this song because we sing it at Camp al...
1952,I love this song for two reasons: 1.it is abou...
1954,Shakira you are so wiredo


If you believe punctuation is just introducing noise to your data, you may want to remove it.

In [9]:
# remove punctuation
text_df_modified = pd.DataFrame(text_df["CONTENT"].str.replace(r'[^\w\s]','', regex=True))
text_df_modified

,CONTENT
0,Huh anyway check out this youtube channel koby...
1,Hey guys check out my new channel and our firs...
2,just for test I have to say murdevcom
3,me shaking my sexy ass on my channel enjoy _
4,watchvvtaRGgvGtWQ Check this out
...,...
1950,well done shakira
1951,I love this song because we sing it at Camp al...
1952,I love this song for two reasons 1it is about ...
1954,Shakira u are so wiredo


If you want to block all YouTubers creating spammy comments, you might want to extract a list of their video IDs

In [10]:
text_df_modified = text_df[text_df["CONTENT"].str.contains(r'watch\?v\=', regex=True)]  # list of spammy comments with YouTube video ids
text_df_videos = text_df_modified["CONTENT"].str.findall(r"watch\?v=(\w+)")  # list of video IDs

# concatenate all the lists in text_df_videos and drop duplicates
text_df_videos = pd.DataFrame(text_df_videos.explode())
text_df_videos = text_df_videos.drop_duplicates()

display(text_df_modified)
display(text_df_videos)

,CONTENT
4,watch?v=vtaRGgvGtWQ Check this out .﻿
33,"Check out my dubstep song ""Fireball"", made wit..."
377,Watch Maroon 5's latest 2nd single from V (It ...
611,Check out my drum cover of E.T. here! thanks -...
700,"<a href=""http://www.youtube.com/watch?v=KQ6zr6..."
730,This Will Always Be My Favorite Song<br />But ...
792,Support the fight for your 4th amendment right...
1005,Youtube comments in a nut shell:<br /><br />.F...
1247,Check out Em&#39;s dope new song monster here:...
1256,youtube.com/watch?v=2ASFn9ShgHk&amp;feature=yo...


,CONTENT
4,vtaRGgvGtWQ
33,telOA6RIO8o
377,TQ046FuAu00
611,NO9pOVZ9OIQ
700,KQ6zr6kCPj8
1247,w6gkM
1256,2ASFn9ShgHk
1281,ARkglzjQuP0
1320,XYTcq5NzMuA
1425,5tu9gN1l310


# Text Tokenization

[`nltk`](https://www.nltk.org/api/nltk.html) is a standard package for Natural Language Processing (NLP).

In [11]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# Let's take one comment
id = 1952

text_df.loc[id]["CONTENT"]

In [ ]:
from nltk.tokenize import word_tokenize  #, sent_tokenize, WhitespaceTokenizer, RegexpTokenizer


tokens_df = text_df.apply(
    lambda x: word_tokenize(x["CONTENT"]),
    axis=1
).to_frame(name="tokens")

tokens_df["tokens"].loc[id]

If you wish to detokenize

In [ ]:
from nltk.tokenize.treebank import TreebankWordDetokenizer

detokenized_df = tokens_df.apply(
    lambda x: TreebankWordDetokenizer().detokenize(x["tokens"]), axis=1).to_frame(name="detokenized")

detokenized_df["detokenized"].loc[id]

# BoW and N-grams

In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

import pandas as pd

Let's say we want to build a spam detector. Let's use Logistic Regression

In [ ]:
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

#model
cv = CountVectorizer( tokenizer=word_tokenize, stop_words='english', lowercase=True) # max_df= 0.95, min_df=0.01, , token_pattern=r"\b\w+\b" OR tokenizer=word_tokenize

# fit
cv.fit(text_df["CONTENT"])

print('number of `tokens`', len(cv.vocabulary_))
cv.vocabulary_

In [ ]:
data_vectorized = cv.transform(text_df["CONTENT"])
data_vectorized_df = pd.DataFrame(data_vectorized.toarray(), columns=cv.get_feature_names_out())
data_vectorized_df

Let's take one comment

In [ ]:
index = 2

text_df.iloc[index]["CONTENT"]

In [ ]:
text_df["CONTENT"].iloc[index]

In [ ]:
data_vectorized_df.iloc[index]

Can we obtain the original text back?

In [ ]:
cv.inverse_transform([data_vectorized_df.iloc[index]])

Train and test a logistic regression model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Split to training and test
X_train, X_test, y_train, y_test = train_test_split(text_df["CONTENT"], class_df["CLASS"], test_size=0.2, stratify=class_df, random_state=42)

# Tokenize and vectorize
cv = CountVectorizer() # max_df= 0.95, min_df=0.01, , token_pattern=r"\b\w+\b" OR tokenizer=word_tokenize , lowercase=True, stop_words='english', ngram_range=(3,5)
X_train = cv.fit_transform(X_train)
X_test = cv.transform(X_test)

# Downstream task
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
pd.DataFrame(confusion_matrix(y_test, y_pred, normalize='true'), columns=model.classes_, index=model.classes_ )

In [ ]:
class_df["CLASS"].value_counts()

##**Questions**

1. Which text preprocessing steps help or harm the classification task?
Discuss the impact of different preprocessing choices on model performance.

2. Do n-gram models with larger values of n outperform a plain Bag-of-Words (BoW) representation?
Explain your answer using both empirical results and theoretical reasoning.

3. What happens when you switch from word-level tokenization to character-level tokenization?
Does this change improve or degrade classification performance? Why?

4. By examining the vectorized table `data_vectorized_df`, which unsupervised learning technique(s) can be used to discover underlying text patterns?
Apply an appropriate method and report the top five most common patterns you find.

5. Does replacing BoW features with some engineered features (e.g., text length, number of exclamation marks, capitalization patterns) lead to better results?

6. Which tokens does logistic regression consider most influential in its classification decisions?
Explain how you identified these tokens and interpret their importance.

# Euclidean vs. Cosine Similarity

Let's compare a couple of messages and see which distance metric is more useful.

In [ ]:
# text_df rows that contain ❤️

mask = (
    text_df["CONTENT"].str.contains("❤", case=False, na=False) |
    text_df["CONTENT"].str.contains("❤️", case=False, na=False)
)

text_df_withhearts = text_df[mask]
text_df_withhearts

Create Text Representation

In [ ]:
from nltk.tokenize import word_tokenize, RegexpTokenizer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# tokenizer
char_tokenizer = RegexpTokenizer(r'.')

#model
cv = CountVectorizer(tokenizer=char_tokenizer.tokenize, lowercase=True) # max_df= 0.95, min_df=0.01, , token_pattern=r"\b\w+\b" OR tokenizer=word_tokenize , stop_words='english'

# fit
text_df_withhearts_tokenized = cv.fit_transform(text_df_withhearts["CONTENT"])

text_df_withhearts_tokenized = pd.DataFrame(text_df_withhearts_tokenized.toarray(), columns=cv.get_feature_names_out())
text_df_withhearts_tokenized

Let's try Euclidean distances

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

euc_dist = euclidean_distances(text_df_withhearts_tokenized)
euc_dist_df = pd.DataFrame(
    euc_dist,
    index=text_df_withhearts.index,
    columns=text_df_withhearts.index
)
euc_dist_df

Let's convert that to a similarity metric

In [ ]:
euc_similarity = 1.0 / (1.0 + euc_dist)
euc_similarity_df = pd.DataFrame(
    euc_similarity,
    index=text_df_withhearts.index,
    columns=text_df_withhearts.index
)
euc_similarity_df

In [ ]:
# Plot histogram of all the values in euc_dist_df
import matplotlib.pyplot as plt

plt.hist(euc_similarity_df.values.flatten(), bins=50)
plt.show()

Let's try cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

cos_sim = np.round(cosine_similarity(text_df_withhearts_tokenized), 2)
cos_sim_df = pd.DataFrame(
    cos_sim,
    index=text_df_withhearts.index,
    columns=text_df_withhearts.index
)
cos_sim_df

In [ ]:
# Plot histogram of all the values in euc_dist_df and stats (mean, std)
import matplotlib.pyplot as plt

plt.hist(cos_sim_df.values.flatten(), bins=50)
plt.show()

# Neural Embeddings

Let's now try a neural embedding approach. Gensim has a [list of pretained models](https://github.com/piskvorky/gensim-data). But, we can also train out own.

In [ ]:
!pip install gensim
!pip install nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize
from gensim.models.word2vec import Word2Vec
import gensim.downloader as api
from gensim.utils import simple_preprocess

# Load the pretrained model
pretrained_model = api.load('glove-wiki-gigaword-50') # or other pretrained models

# or train your own Word2Vec on text_df["CONTENT"].
# corpus = text_df["CONTENT"].tolist()
# corpus = [word_tokenize(text) for text in corpus]
# # corpus = [simple_preprocess(text) for text in corpus] #You may optionally tokenize and clean up
# word2vec_trained_model = Word2Vec(sentences=corpus) # Take a look at the arguments you have

In [ ]:
#The model's vocabulary
pretrained_model.key_to_index
# word2vec_trained_model.wv.key_to_index

In [ ]:
# A word's embedding
pretrained_model["hi"]
# word2vec_trained_model.wv["hi"]

In [ ]:
# Get a sentence embedding
sentence = text_df["CONTENT"].iloc[0]

pretrained_model.get_mean_vector(sentence, ignore_missing=True)
# word2vec_trained_model.wv.get_mean_vector(sentence, ignore_missing=True)

In [ ]:
# Get similarity score
sentence1 = text_df["CONTENT"].iloc[0]
sentence2 = text_df["CONTENT"].iloc[1]

sim_score = pretrained_model.n_similarity(word_tokenize(sentence1), word_tokenize(sentence2))
# sim_score = word2vec_trained_model.wv.n_similarity(word_tokenize(sentence1), word_tokenize(sentence2))

print(sentence1)
print(sentence2)
print("similarity score:", sim_score)

### Spam classification with neural embeddings

Let's perform spam calssification using the neural embeddings

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(text_df["CONTENT"].tolist(), class_df["CLASS"], test_size=0.2, stratify=class_df["CLASS"], random_state=42)

# X_train = [word_tokenize(text) for text in X_train]
# X_test = [word_tokenize(text) for text in X_test]
#You may optionally tokenize and clean up
X_train = [simple_preprocess(text) for text in X_train]
X_test  = [simple_preprocess(text) for text in X_test]

# embedding_model = Word2Vec(sentences=X_train, epochs=50).wv # Take a look at the arguments you have such as vector_size min_count epochs
embedding_model = pretrained_model

X_train = [embedding_model.get_mean_vector(text, ignore_missing=True) if text else np.zeros(embedding_model.vector_size) for text in X_train]
X_test = [embedding_model.get_mean_vector(text, ignore_missing=True) if text else np.zeros(embedding_model.vector_size) for text in X_test]

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")
pd.DataFrame(confusion_matrix(y_test, y_pred, normalize='true'), columns=model.classes_, index=model.classes_ )

##Questions:

- Plot and compare the word embeddings of different models (BoW, pre-trained, word2vec, etc.). Record the differences and any interesting observations based on your visualization.
- Initially, BoW representation yields better spam classification performance than neural embeddings. Is there anything you could do to boost the performance of a neural-embedding-based classifier? If yes, enumerate the modifications. If no, what is your explanation?
- Would dimensionality reduction help improve the classifier's performance?
- What would text message clustering look like? What do the resulting clusters represent?

## Pretrained Deep Learning Embedding

Let's try now more advanced deep learning models that produce more sophisticated embeddings that take word order into account.

Let's use a pre-trained model from [`SentenceTransformer`](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html).


In [ ]:
!pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = text_df["CONTENT"].loc[[1950, 1954, 1955, 1951, 2]]
print(sentences)

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences.tolist())
print(embeddings.shape)

In [ ]:
# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)